# 🔬 05. Model Interpretability & Explainability with SHAP
**Project**: XGBoost-Powered PE Malware Detection  
**Purpose**: Uncover feature attribution, decision waterfalls, and interaction values using TreeSHAP.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('../').resolve()))
from utils.preprocessing import MalwarePreprocessor, prepare_splits, compute_scale_pos_weight, ALL_FEATURES
from utils.shap_explainer import MalwareSHAPExplainer
from models.xgboost_model import XGBoostMalwareDetector

df = pd.read_csv('../data/synthetic_malware_data.csv')
train_df, val_df, test_df = prepare_splits(df)

preprocessor = MalwarePreprocessor(scaler_type='robust')
X_train = preprocessor.fit_transform(train_df)
X_test = preprocessor.transform(test_df)
y_train = train_df['label'].values
y_test = test_df['label'].values
spw = compute_scale_pos_weight(train_df['label'])

model = XGBoostMalwareDetector(feature_names=ALL_FEATURES, scale_pos_weight=spw)
model.fit(X_train, y_train)


## 1. Global Feature Importance via TreeSHAP


In [ ]:
explainer = MalwareSHAPExplainer(model.model, feature_names=ALL_FEATURES)
explainer.fit(X_train[:200])

sample_test = X_test[:300]
shap_values = explainer.shap_values(sample_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, sample_test, feature_names=ALL_FEATURES, show=False)
plt.title('SHAP Beeswarm Summary Plot', fontsize=14)
plt.tight_layout()
plt.show()


## 2. Local Sample Explanation (Waterfall)
Explaining why an individual suspicious file was flagged as malware.


In [ ]:
malware_idx = np.where(y_test == 1)[0][0]
waterfall_data = explainer.get_waterfall_data(X_test[malware_idx])

plt.figure(figsize=(10, 6))
plt.barh(waterfall_data['feature_names'][:10], waterfall_data['shap_values'][:10], color='#ff007f')
plt.title(f'Top 10 SHAP Contributions for Malware Sample (Base: {waterfall_data["base_value"]:.2f})')
plt.xlabel('SHAP value (Impact on log-odds output)')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()
